In [ ]:
#1)VectorStoreRetriever	Vector-based	Embedding similarity search	General-purpose RAG
# ConversationalRetrievalChain

# 1️⃣ Imports
from langchain_ollama import ChatOllama
from langchain_classic.tools import StructuredTool
from langchain_classic.agents import initialize_agent, AgentType
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from langchain_classic.chains import RetrievalQA, ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory
from langchain_text_splitters import CharacterTextSplitter

# 1. Setup

llm = ChatOllama(
    model='mistral',
    temperature=0)

# 2. Vector DB
with open("sample.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

# 🧠 Split the text into smaller chunks
splitter = CharacterTextSplitter(separator="\n", 
                                 chunk_size=300, 
                                 chunk_overlap=50)
texts = splitter.split_text(text_data)

embedding = OllamaEmbeddings(
        model="nomic-embed-text")
vectorstore = FAISS.from_texts(texts, embedding)
retriever = vectorstore.as_retriever()

# 3. Conversational RAG chain
rag_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    return_source_documents=False
)

# 4. Memory for chat history
memory = ConversationBufferMemory(memory_key="chat_history")

# 5. Wrap RAG as a StructuredTool
def rag_tool_fn(question: str) -> str:
    return rag_chain.invoke({
        "question": question,
        "chat_history": []
    })["answer"]

rag_tool = StructuredTool.from_function(
    name="RAG_QA",
    description="Use this to answer questions about LangChain.",
    func=rag_tool_fn
)

# 6. Create agent
agent = initialize_agent(
    tools=[rag_tool],
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True,
    memory=memory,
    handle_parsing_errors=True
)

# 7. Run conversation
res1 = agent.invoke("What is LangChain?")
res2 = agent.invoke("Who created it?")
res3 = agent.invoke("Explain LangChain and LLM simply in 3 bullet points.")

print("1️⃣ First question")
print("Answer:", res1['output'])

print("\n2️⃣ Follow-up")
print("Answer:", res2['output'])

print("\n3️⃣ Summarize")
print("Answer:", res3['output'])


Created a chunk of size 622, which is longer than the specified 300
Created a chunk of size 803, which is longer than the specified 300




> Entering new AgentExecutor chain...
 Thought: Do I need to use a tool? Yes
Action: RAG_QA
Action Input: What is LangChain?
Observation:  LangChain is an open-source framework created by Harrison Chase that bridges the gap between static Large Language Models (LLMs) and dynamic, data-aware applications. It allows developers to create sophisticated workflows by chaining together different components such as prompt templates, memory modules, and document loaders. By using LangChain, a project can implement Retrieval-Augmented Generation (RAG), where the LLM queries a private database or the web before generating an answer, minimizing "hallucinations" and ensuring the AI's output is grounded in factual, up-to-date information. Ankush Gola is another co-founder of LangChain who previously worked as a machine learning engineer at Meta (Facebook) and Robust Intelligence.
Thought: Do I need to use a tool? No
AI: LangChain is an open-source framework that bridges the gap between static Larg